<a href="https://colab.research.google.com/github/mikerich135/credit-risk-modeling/blob/xgboost_by_richard/eda_feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:

# Install dependencies
!pip install scikit-learn
!pip install pandas
!pip install numpy
!pip install matplotlib

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
#import scikit-learn as ski
from google.colab import drive

def aggregate_bureau(bureau_path, bureau_balance_path):
    bureau = pd.read_csv(bureau_path)
    bb = pd.read_csv(bureau_balance_path)

    # First: aggregate bureau_balance to bureau level
    bb_agg = bb.groupby('SK_ID_BUREAU').agg(
        MONTHS_BALANCE_min=('MONTHS_BALANCE', 'min'),
        MONTHS_BALANCE_max=('MONTHS_BALANCE', 'max'),
        MONTHS_BALANCE_size=('MONTHS_BALANCE', 'size')
    ).reset_index()

    # Status flags from bureau_balance
    status_dummies = pd.get_dummies(bb['STATUS'], prefix='STATUS')
    bb_status = pd.concat([bb[['SK_ID_BUREAU']], status_dummies], axis=1)
    bb_status_agg = bb_status.groupby('SK_ID_BUREAU').mean().reset_index()

    bureau = bureau.merge(bb_agg, on='SK_ID_BUREAU', how='left')
    bureau = bureau.merge(bb_status_agg, on='SK_ID_BUREAU', how='left')

    # Now aggregate bureau to SK_ID_CURR level
    num_aggs = {
        'DAYS_CREDIT': ['min', 'max', 'mean', 'var'],
        'DAYS_CREDIT_ENDDATE': ['min', 'max', 'mean'],
        'CREDIT_DAY_OVERDUE': ['max', 'mean'],
        'AMT_CREDIT_MAX_OVERDUE': ['mean', 'max'],
        'AMT_CREDIT_SUM': ['mean', 'sum', 'max'],
        'AMT_CREDIT_SUM_DEBT': ['mean', 'sum'],
        'AMT_CREDIT_SUM_OVERDUE': ['mean', 'sum'],
        'CNT_CREDIT_PROLONG': ['sum'],
    }

    bureau_agg = bureau.groupby('SK_ID_CURR').agg(num_aggs)
    bureau_agg.columns = ['BURO_' + '_'.join(c).upper() for c in bureau_agg.columns]
    bureau_agg['BURO_COUNT'] = bureau.groupby('SK_ID_CURR').size()

    # Active credits only
    active = bureau[bureau['CREDIT_ACTIVE'] == 'Active']
    active_agg = active.groupby('SK_ID_CURR').agg({
        'AMT_CREDIT_SUM': ['sum', 'mean'],
        'AMT_CREDIT_SUM_DEBT': ['sum', 'mean'],
    })
    active_agg.columns = ['BURO_ACTIVE_' + '_'.join(c).upper() for c in active_agg.columns]

    return bureau_agg.join(active_agg, how='left').reset_index()

 #── Helper: build aggregation for a subset ────────────────────────────────────
def aggregate_subset(subset: pd.DataFrame, prefix: str) -> pd.DataFrame:
    """
    Aggregate a filtered DataFrame by SK_ID_CURR.
    prefix  : column name prefix, e.g. 'APPROVED' or 'CANCELLED'
    """
    agg = subset.groupby("SK_ID_CURR").agg(
        # ── Count ──────────────────────────────────────────────────
        count                  = ("SK_ID_PREV",              "count"),

        # ── Credit amount ──────────────────────────────────────────
        total_credit           = ("AMT_CREDIT",              "sum"),
        avg_credit             = ("AMT_CREDIT",              "mean"),
        max_credit             = ("AMT_CREDIT",              "max"),
        min_credit             = ("AMT_CREDIT",              "min"),

        # ── Application amount ─────────────────────────────────────
        total_application      = ("AMT_APPLICATION",         "sum"),
        avg_application        = ("AMT_APPLICATION",         "mean"),

        # ── Annuity ────────────────────────────────────────────────
        total_annuity          = ("AMT_ANNUITY",             "sum"),
        avg_annuity            = ("AMT_ANNUITY",             "mean"),

        # ── Down payment ───────────────────────────────────────────
        avg_down_payment       = ("AMT_DOWN_PAYMENT",        "mean"),

        # ── Goods price ────────────────────────────────────────────
        avg_goods_price        = ("AMT_GOODS_PRICE",         "mean"),

        # ── Credit-to-application ratio (mean across apps) ─────────
        # computed below after agg

        # ── Loan term ──────────────────────────────────────────────
        avg_cnt_payment        = ("CNT_PAYMENT",             "mean"),
        max_cnt_payment        = ("CNT_PAYMENT",             "max"),

        # ── Days since decision (negative = days before today) ─────
        avg_days_decision      = ("DAYS_DECISION",           "mean"),
        min_days_decision      = ("DAYS_DECISION",           "min"),   # most recent

        # ── Insurance flag ─────────────────────────────────────────
        sum_insured            = ("NFLAG_INSURED_ON_APPROVAL","sum"),

        # ── Interest rates (may be mostly NaN) ─────────────────────
        avg_rate_down_payment  = ("RATE_DOWN_PAYMENT",       "mean"),
        avg_rate_interest_prim = ("RATE_INTEREST_PRIMARY",   "mean"),
    ).reset_index()

    # Credit-to-application ratio per customer
    agg["credit_to_application_ratio"] = (
        agg["total_credit"] / agg["total_application"].replace(0, np.nan)
    )

    # Add prefix to all feature columns (keep SK_ID_CURR clean)
    agg.columns = (
        ["SK_ID_CURR"]
        + [f"{prefix}_{col.upper()}" for col in agg.columns[1:]]
    )

    return agg

def aggregate_pre_application(filepath: str):

  df = pd.read_csv(filepath)
  # ── Split by status ────────────────────────────────────────────────────────────
  approved  = df[df["NAME_CONTRACT_STATUS"] == "Approved"]
  cancelled = df[df["NAME_CONTRACT_STATUS"] == "Canceled"]

  print(f"Approved rows  : {len(approved):,}")
  print(f"Cancelled rows : {len(cancelled):,}\n")

  # ── Aggregate each group ───────────────────────────────────────────────────────
  print("Aggregating approved applications ...")
  agg_approved  = aggregate_subset(approved,  prefix="APPROVED")

  print("Aggregating cancelled applications ...")
  agg_cancelled = aggregate_subset(cancelled, prefix="CANCELLED")

  # ── Merge on SK_ID_CURR (outer join — keep all customers) ─────────────────────
  print("\nMerging approved & cancelled features ...")
  agg_final = pd.merge(agg_approved, agg_cancelled, on="SK_ID_CURR", how="outer")

  # Fill NaN with 0 for count/sum columns; leave ratio/avg as NaN (meaningful absence)
  count_sum_cols = [
      c for c in agg_final.columns
      if any(c.endswith(s) for s in ("_COUNT", "_TOTAL_CREDIT", "_TOTAL_APPLICATION",
                                      "_TOTAL_ANNUITY", "_SUM_INSURED"))
  ]
  agg_final[count_sum_cols] = agg_final[count_sum_cols].fillna(0)

  print(f"\nFinal aggregated shape : {agg_final.shape}")
  print(f"Customers covered      : {agg_final['SK_ID_CURR'].nunique():,}")
  print(f"\nColumns:\n{agg_final.columns.tolist()}\n")
  print(agg_final.head(3).to_string())
  return agg_final


def pos_cash_aggregate(filepath: str):
    df = pd.read_csv(filepath)
    # Instalments completed so far for each row
    df["CNT_INSTALMENT_COMPLETED"] = (
        df["CNT_INSTALMENT"] - df["CNT_INSTALMENT_FUTURE"]
    )

    # Binary: was this month ever overdue?
    df["DPD_FLAG"]     = (df["SK_DPD"]     > 0).astype(int)
    df["DPD_DEF_FLAG"] = (df["SK_DPD_DEF"] > 0).astype(int)

    # Binary: is this entry an Active contract?
    df["IS_ACTIVE"]    = (df["NAME_CONTRACT_STATUS"] == "Active").astype(int)
    df["IS_COMPLETED"] = (df["NAME_CONTRACT_STATUS"] == "Completed").astype(int)

    # ── Recent window (last 12 months) ────────────────────────────────────────────
    recent = df[df["MONTHS_BALANCE"] >= -12]

    # ── Main aggregation by SK_ID_CURR ────────────────────────────────────────────
    print("Aggregating all history ...")
    agg_all = df.groupby("SK_ID_CURR").agg(

        # ── Loan count & history length ───────────────────────────────
        POS_NUM_LOANS           = ("SK_ID_PREV",              pd.Series.nunique),
        POS_NUM_ENTRIES         = ("SK_ID_PREV",              "count"),
        POS_MONTHS_BALANCE_MIN  = ("MONTHS_BALANCE",          "min"),   # earliest record
        POS_MONTHS_BALANCE_MAX  = ("MONTHS_BALANCE",          "max"),   # most recent record
        POS_MONTHS_BALANCE_MEAN = ("MONTHS_BALANCE",          "mean"),

        # ── Instalment progress ───────────────────────────────────────
        POS_CNT_INSTALMENT_MEAN        = ("CNT_INSTALMENT",           "mean"),
        POS_CNT_INSTALMENT_MAX         = ("CNT_INSTALMENT",           "max"),
        POS_CNT_INSTALMENT_FUTURE_MEAN = ("CNT_INSTALMENT_FUTURE",    "mean"),
        POS_CNT_INSTALMENT_FUTURE_SUM  = ("CNT_INSTALMENT_FUTURE",    "sum"),
        POS_CNT_INSTALMENT_COMPLETED_SUM  = ("CNT_INSTALMENT_COMPLETED", "sum"),
        POS_CNT_INSTALMENT_COMPLETED_MEAN = ("CNT_INSTALMENT_COMPLETED","mean"),

        # ── Days past due (DPD) ───────────────────────────────────────
        POS_SK_DPD_MAX          = ("SK_DPD",      "max"),
        POS_SK_DPD_MEAN         = ("SK_DPD",      "mean"),
        POS_SK_DPD_SUM          = ("SK_DPD",      "sum"),
        POS_SK_DPD_DEF_MAX      = ("SK_DPD_DEF",  "max"),
        POS_SK_DPD_DEF_MEAN     = ("SK_DPD_DEF",  "mean"),
        POS_SK_DPD_DEF_SUM      = ("SK_DPD_DEF",  "sum"),

        # ── Overdue flags: how many months had any overdue? ───────────
        POS_DPD_FLAG_SUM        = ("DPD_FLAG",     "sum"),
        POS_DPD_FLAG_MEAN       = ("DPD_FLAG",     "mean"),   # overdue rate
        POS_DPD_DEF_FLAG_SUM    = ("DPD_DEF_FLAG", "sum"),
        POS_DPD_DEF_FLAG_MEAN   = ("DPD_DEF_FLAG", "mean"),

        # ── Contract status counts ────────────────────────────────────
        POS_IS_ACTIVE_SUM       = ("IS_ACTIVE",    "sum"),
        POS_IS_COMPLETED_SUM    = ("IS_COMPLETED", "sum"),

    ).reset_index()

    # ── Recent-window aggregation (last 12 months) ────────────────────────────────
    print("Aggregating last-12-months window ...")
    agg_recent = recent.groupby("SK_ID_CURR").agg(
        POS_RECENT_NUM_ENTRIES       = ("SK_ID_PREV",    "count"),
        POS_RECENT_DPD_MAX          = ("SK_DPD",        "max"),
        POS_RECENT_DPD_MEAN         = ("SK_DPD",        "mean"),
        POS_RECENT_DPD_SUM          = ("SK_DPD",        "sum"),
        POS_RECENT_DPD_DEF_MAX      = ("SK_DPD_DEF",    "max"),
        POS_RECENT_DPD_DEF_MEAN     = ("SK_DPD_DEF",    "mean"),
        POS_RECENT_DPD_FLAG_SUM     = ("DPD_FLAG",      "sum"),
        POS_RECENT_DPD_FLAG_MEAN    = ("DPD_FLAG",      "mean"),
        POS_RECENT_DPD_DEF_FLAG_SUM = ("DPD_DEF_FLAG",  "sum"),
        POS_RECENT_INSTALMENT_FUTURE_MEAN = ("CNT_INSTALMENT_FUTURE", "mean"),
    ).reset_index()

    # ── Merge all + recent ────────────────────────────────────────────────────────
    print("Merging all-history + recent-window features ...")
    agg_final = pd.merge(agg_all, agg_recent, on="SK_ID_CURR", how="left")

    # ── Ratio features ────────────────────────────────────────────────────────────
    # What fraction of months were overdue? (recent vs all-time)
    agg_final["POS_DPD_RECENT_VS_ALL_RATIO"] = (
        agg_final["POS_RECENT_DPD_FLAG_SUM"]
        / agg_final["POS_DPD_FLAG_SUM"].replace(0, np.nan)
    )

    # How much of total instalments are still remaining?
    agg_final["POS_INSTALMENT_COMPLETION_RATE"] = (
        agg_final["POS_CNT_INSTALMENT_COMPLETED_SUM"]
        / (agg_final["POS_CNT_INSTALMENT_COMPLETED_SUM"]
          + agg_final["POS_CNT_INSTALMENT_FUTURE_SUM"]).replace(0, np.nan)
    )

    # ── Summary ───────────────────────────────────────────────────────────────────
    print(f"\n{'─'*55}")
    print(f"  Final shape      : {agg_final.shape}")
    print(f"  Customers        : {agg_final['SK_ID_CURR'].nunique():,}")
    print(f"  Features created : {agg_final.shape[1] - 1}")
    print(f"{'─'*55}")
    print(f"\n  Columns:\n")
    for col in agg_final.columns:
        print(f"    {col}")

    return agg_final

def installment_payments_aggregate(filepath: str):
    df = pd.read_csv(filepath)
    df["PAYMENT_DELAY"] = df["DAYS_ENTRY_PAYMENT"] - df["DAYS_INSTALMENT"]

    # Payment shortfall: positive = underpaid, negative = overpaid
    df["PAYMENT_DIFF"] = df["AMT_INSTALMENT"] - df["AMT_PAYMENT"]

    # Payment ratio: how much of the due amount was actually paid
    df["PAYMENT_RATIO"] = df["AMT_PAYMENT"] / df["AMT_INSTALMENT"].replace(0, np.nan)

    # Binary flags
    df["IS_LATE"]        = (df["PAYMENT_DELAY"] > 0).astype(int)
    df["IS_EARLY"]       = (df["PAYMENT_DELAY"] < 0).astype(int)
    df["IS_UNDERPAID"]   = (df["PAYMENT_DIFF"]  > 0).astype(int)
    df["IS_OVERPAID"]    = (df["PAYMENT_DIFF"]  < 0).astype(int)

    # ── Recent window (last 12 months = last 365 days) ────────────────────────────
    # DAYS_ENTRY_PAYMENT is negative; closer to 0 = more recent
    recent = df[df["DAYS_ENTRY_PAYMENT"] >= -365]

    # ── Main aggregation by SK_ID_CURR ────────────────────────────────────────────
    print("Aggregating all-history features ...")
    agg_all = df.groupby("SK_ID_CURR").agg(

        # ── Loan & instalment counts ──────────────────────────────────
        INSTAL_NUM_LOANS           = ("SK_ID_PREV",             pd.Series.nunique),
        INSTAL_NUM_ENTRIES         = ("SK_ID_PREV",             "count"),
        INSTAL_NUM_INSTALMENT_MAX  = ("NUM_INSTALMENT_NUMBER",  "max"),   # furthest instalment reached

        # ── Due amount ────────────────────────────────────────────────
        INSTAL_AMT_INSTALMENT_SUM  = ("AMT_INSTALMENT",  "sum"),
        INSTAL_AMT_INSTALMENT_MEAN = ("AMT_INSTALMENT",  "mean"),
        INSTAL_AMT_INSTALMENT_MAX  = ("AMT_INSTALMENT",  "max"),
        INSTAL_AMT_INSTALMENT_MIN  = ("AMT_INSTALMENT",  "min"),

        # ── Actual payment ────────────────────────────────────────────
        INSTAL_AMT_PAYMENT_SUM     = ("AMT_PAYMENT",     "sum"),
        INSTAL_AMT_PAYMENT_MEAN    = ("AMT_PAYMENT",     "mean"),
        INSTAL_AMT_PAYMENT_MAX     = ("AMT_PAYMENT",     "max"),
        INSTAL_AMT_PAYMENT_MIN     = ("AMT_PAYMENT",     "min"),

        # ── Payment shortfall (due - paid) ────────────────────────────
        INSTAL_PAYMENT_DIFF_SUM    = ("PAYMENT_DIFF",    "sum"),
        INSTAL_PAYMENT_DIFF_MEAN   = ("PAYMENT_DIFF",    "mean"),
        INSTAL_PAYMENT_DIFF_MAX    = ("PAYMENT_DIFF",    "max"),  # worst underpayment

        # ── Payment ratio (paid / due) ────────────────────────────────
        INSTAL_PAYMENT_RATIO_MEAN  = ("PAYMENT_RATIO",   "mean"),
        INSTAL_PAYMENT_RATIO_MIN   = ("PAYMENT_RATIO",   "min"),  # worst payment coverage

        # ── Payment delay (days late) ─────────────────────────────────
        INSTAL_PAYMENT_DELAY_SUM   = ("PAYMENT_DELAY",   "sum"),
        INSTAL_PAYMENT_DELAY_MEAN  = ("PAYMENT_DELAY",   "mean"),
        INSTAL_PAYMENT_DELAY_MAX   = ("PAYMENT_DELAY",   "max"),  # worst single delay
        INSTAL_PAYMENT_DELAY_MIN   = ("PAYMENT_DELAY",   "min"),  # most early payment

        # ── Late / early / underpaid flags ────────────────────────────
        INSTAL_LATE_COUNT          = ("IS_LATE",         "sum"),
        INSTAL_LATE_RATE           = ("IS_LATE",         "mean"),   # % of payments late
        INSTAL_EARLY_COUNT         = ("IS_EARLY",        "sum"),
        INSTAL_EARLY_RATE          = ("IS_EARLY",        "mean"),
        INSTAL_UNDERPAID_COUNT     = ("IS_UNDERPAID",    "sum"),
        INSTAL_UNDERPAID_RATE      = ("IS_UNDERPAID",    "mean"),   # % of payments underpaid
        INSTAL_OVERPAID_COUNT      = ("IS_OVERPAID",     "sum"),

        # ── Days instalment (how far back does history go) ────────────
        INSTAL_DAYS_INSTALMENT_MIN = ("DAYS_INSTALMENT", "min"),   # oldest due date
        INSTAL_DAYS_INSTALMENT_MAX = ("DAYS_INSTALMENT", "max"),   # most recent due date

    ).reset_index()

    # ── Recent-window aggregation (last 12 months) ────────────────────────────────
    print("Aggregating last-12-months window ...")
    agg_recent = recent.groupby("SK_ID_CURR").agg(
        INSTAL_RECENT_NUM_ENTRIES      = ("SK_ID_PREV",      "count"),
        INSTAL_RECENT_AMT_PAYMENT_SUM  = ("AMT_PAYMENT",     "sum"),
        INSTAL_RECENT_AMT_PAYMENT_MEAN = ("AMT_PAYMENT",     "mean"),
        INSTAL_RECENT_PAYMENT_DIFF_MEAN= ("PAYMENT_DIFF",    "mean"),
        INSTAL_RECENT_PAYMENT_DIFF_MAX = ("PAYMENT_DIFF",    "max"),
        INSTAL_RECENT_PAYMENT_RATIO_MEAN=("PAYMENT_RATIO",   "mean"),
        INSTAL_RECENT_DELAY_MEAN       = ("PAYMENT_DELAY",   "mean"),
        INSTAL_RECENT_DELAY_MAX        = ("PAYMENT_DELAY",   "max"),
        INSTAL_RECENT_LATE_COUNT       = ("IS_LATE",         "sum"),
        INSTAL_RECENT_LATE_RATE        = ("IS_LATE",         "mean"),
        INSTAL_RECENT_UNDERPAID_COUNT  = ("IS_UNDERPAID",    "sum"),
        INSTAL_RECENT_UNDERPAID_RATE   = ("IS_UNDERPAID",    "mean"),
    ).reset_index()

    # Rename to avoid collision
    agg_recent.columns = ["SK_ID_CURR"] + [
        c for c in agg_recent.columns[1:]   # already prefixed above
    ]

    # ── Merge all-history + recent ────────────────────────────────────────────────
    print("Merging all-history + recent-window features ...")
    agg_final = pd.merge(agg_all, agg_recent, on="SK_ID_CURR", how="left")

    # ── Extra ratio features ──────────────────────────────────────────────────────
    # Did they pay back more or less in total vs what was owed?
    agg_final["INSTAL_TOTAL_PAYMENT_RATIO"] = (
        agg_final["INSTAL_AMT_PAYMENT_SUM"]
        / agg_final["INSTAL_AMT_INSTALMENT_SUM"].replace(0, np.nan)
    )

    # Is recent late rate worse than all-time? (deterioration signal)
    agg_final["INSTAL_LATE_RATE_RECENT_VS_ALL"] = (
        agg_final["INSTAL_RECENT_LATE_RATE"]
        / agg_final["INSTAL_LATE_RATE"].replace(0, np.nan)
    )

    # ── Summary ───────────────────────────────────────────────────────────────────
    print(f"\n{'─'*55}")
    print(f"  Final shape      : {agg_final.shape}")
    print(f"  Customers        : {agg_final['SK_ID_CURR'].nunique():,}")
    print(f"  Features created : {agg_final.shape[1] - 1}")
    print(f"{'─'*55}")
    print(f"\n  All columns:")
    for col in agg_final.columns:
        print(f"    {col}")

    print(f"\n  Sample (3 rows):")
    print(agg_final.head(3).T.to_string())
    return agg_final

def credit_balance_aggregate(filepath: str):
    df = pd.read_csv(filepath)
    # Credit utilisation: how much of the limit is being used (0 to 1+)
    df["CREDIT_UTILISATION"] = (
        df["AMT_BALANCE"] / df["AMT_CREDIT_LIMIT_ACTUAL"].replace(0, np.nan)
    )

    # Minimum payment coverage: did they pay at least the minimum due?
    df["MIN_PAYMENT_RATIO"] = (
        df["AMT_PAYMENT_TOTAL_CURRENT"]
        / df["AMT_INST_MIN_REGULARITY"].replace(0, np.nan)
    )

    # Payment vs balance ratio: how aggressively are they paying down?
    df["PAYMENT_TO_BALANCE_RATIO"] = (
        df["AMT_PAYMENT_TOTAL_CURRENT"]
        / df["AMT_BALANCE"].replace(0, np.nan)
    )

    # Total receivable gap: difference between total receivable and principal
    df["RECEIVABLE_TO_PRINCIPAL_DIFF"] = (
        df["AMT_TOTAL_RECEIVABLE"] - df["AMT_RECEIVABLE_PRINCIPAL"]
    )

    # Binary flags
    df["DPD_FLAG"]          = (df["SK_DPD"]     > 0).astype(int)
    df["DPD_DEF_FLAG"]      = (df["SK_DPD_DEF"] > 0).astype(int)
    df["IS_ACTIVE"]         = (df["NAME_CONTRACT_STATUS"] == "Active").astype(int)
    df["HIGH_UTILISATION"]  = (df["CREDIT_UTILISATION"] > 0.8).astype(int)  # >80% used

    # ── Recent window (last 12 months) ────────────────────────────────────────────
    recent = df[df["MONTHS_BALANCE"] >= -12]

    # ── Main aggregation by SK_ID_CURR ────────────────────────────────────────────
    print("Aggregating all-history features ...")
    agg_all = df.groupby("SK_ID_CURR").agg(

        # ── Card & history counts ─────────────────────────────────────
        CC_NUM_CARDS              = ("SK_ID_PREV",                  pd.Series.nunique),
        CC_NUM_ENTRIES            = ("SK_ID_PREV",                  "count"),
        CC_MONTHS_BALANCE_MIN     = ("MONTHS_BALANCE",              "min"),
        CC_MONTHS_BALANCE_MAX     = ("MONTHS_BALANCE",              "max"),

        # ── Balance ───────────────────────────────────────────────────
        CC_AMT_BALANCE_MEAN       = ("AMT_BALANCE",                 "mean"),
        CC_AMT_BALANCE_MAX        = ("AMT_BALANCE",                 "max"),
        CC_AMT_BALANCE_SUM        = ("AMT_BALANCE",                 "sum"),
        CC_AMT_BALANCE_MIN        = ("AMT_BALANCE",                 "min"),

        # ── Credit limit ──────────────────────────────────────────────
        CC_CREDIT_LIMIT_MEAN      = ("AMT_CREDIT_LIMIT_ACTUAL",     "mean"),
        CC_CREDIT_LIMIT_MAX       = ("AMT_CREDIT_LIMIT_ACTUAL",     "max"),

        # ── Credit utilisation ────────────────────────────────────────
        CC_UTILISATION_MEAN       = ("CREDIT_UTILISATION",          "mean"),
        CC_UTILISATION_MAX        = ("CREDIT_UTILISATION",          "max"),
        CC_HIGH_UTILISATION_SUM   = ("HIGH_UTILISATION",            "sum"),
        CC_HIGH_UTILISATION_RATE  = ("HIGH_UTILISATION",            "mean"),

        # ── Drawings (total spending) ─────────────────────────────────
        CC_AMT_DRAWINGS_SUM        = ("AMT_DRAWINGS_CURRENT",        "sum"),
        CC_AMT_DRAWINGS_MEAN      = ("AMT_DRAWINGS_CURRENT",        "mean"),
        CC_AMT_DRAWINGS_ATM_SUM   = ("AMT_DRAWINGS_ATM_CURRENT",    "sum"),
        CC_AMT_DRAWINGS_ATM_MEAN  = ("AMT_DRAWINGS_ATM_CURRENT",    "mean"),
        CC_AMT_DRAWINGS_POS_SUM   = ("AMT_DRAWINGS_POS_CURRENT",    "sum"),
        CC_CNT_DRAWINGS_MEAN      = ("CNT_DRAWINGS_CURRENT",        "mean"),
        CC_CNT_DRAWINGS_ATM_MEAN  = ("CNT_DRAWINGS_ATM_CURRENT",    "mean"),
        CC_CNT_DRAWINGS_POS_MEAN  = ("CNT_DRAWINGS_POS_CURRENT",    "mean"),

        # ── Payments ──────────────────────────────────────────────────
        CC_AMT_PAYMENT_TOTAL_SUM  = ("AMT_PAYMENT_TOTAL_CURRENT",   "sum"),
        CC_AMT_PAYMENT_TOTAL_MEAN = ("AMT_PAYMENT_TOTAL_CURRENT",   "mean"),
        CC_AMT_PAYMENT_TOTAL_MAX  = ("AMT_PAYMENT_TOTAL_CURRENT",   "max"),
        CC_AMT_PAYMENT_CURRENT_MEAN = ("AMT_PAYMENT_CURRENT",       "mean"),

        # ── Minimum payment behaviour ─────────────────────────────────
        CC_MIN_PAYMENT_RATIO_MEAN = ("MIN_PAYMENT_RATIO",           "mean"),
        CC_MIN_PAYMENT_RATIO_MIN  = ("MIN_PAYMENT_RATIO",           "min"),
        CC_AMT_INST_MIN_MEAN      = ("AMT_INST_MIN_REGULARITY",     "mean"),

        # ── Payment to balance ratio ──────────────────────────────────
        CC_PAYMENT_TO_BAL_MEAN    = ("PAYMENT_TO_BALANCE_RATIO",    "mean"),

        # ── Receivables ───────────────────────────────────────────────
        CC_AMT_RECEIVABLE_MEAN    = ("AMT_RECIVABLE",               "mean"),
        CC_AMT_TOTAL_RECV_MEAN    = ("AMT_TOTAL_RECEIVABLE",        "mean"),
        CC_RECV_PRINCIPAL_MEAN    = ("AMT_RECEIVABLE_PRINCIPAL",    "mean"),
        CC_RECV_TO_PRINCIPAL_DIFF = ("RECEIVABLE_TO_PRINCIPAL_DIFF","mean"),

        # ── Instalment maturity ───────────────────────────────────────
        CC_CNT_INSTALMENT_MAX     = ("CNT_INSTALMENT_MATURE_CUM",   "max"),
        CC_CNT_INSTALMENT_MEAN    = ("CNT_INSTALMENT_MATURE_CUM",   "mean"),

        # ── DPD (days past due) ───────────────────────────────────────
        CC_SK_DPD_MAX             = ("SK_DPD",                      "max"),
        CC_SK_DPD_MEAN            = ("SK_DPD",                      "mean"),
        CC_SK_DPD_SUM             = ("SK_DPD",                      "sum"),
        CC_SK_DPD_DEF_MAX         = ("SK_DPD_DEF",                  "max"),
        CC_SK_DPD_DEF_MEAN         = ("SK_DPD_DEF",                  "mean"),

        # ── DPD flags ─────────────────────────────────────────────────
        CC_DPD_FLAG_SUM           = ("DPD_FLAG",                    "sum"),
        CC_DPD_FLAG_MEAN          = ("DPD_FLAG",                    "mean"),   # overdue rate
        CC_DPD_DEF_FLAG_SUM       = ("DPD_DEF_FLAG",                "sum"),
        CC_DPD_DEF_FLAG_MEAN      = ("DPD_DEF_FLAG",                "mean"),

        # ── Contract status ───────────────────────────────────────────
        CC_IS_ACTIVE_SUM          = ("IS_ACTIVE",                   "sum"),

    ).reset_index()

    # ── Recent-window aggregation (last 12 months) ────────────────────────────────
    print("Aggregating last-12-months window ...")
    agg_recent = recent.groupby("SK_ID_CURR").agg(
        CC_RECENT_NUM_ENTRIES          = ("SK_ID_PREV",               "count"),
        CC_RECENT_AMT_BALANCE_MEAN     = ("AMT_BALANCE",              "mean"),
        CC_RECENT_AMT_BALANCE_MAX      = ("AMT_BALANCE",              "max"),
        CC_RECENT_UTILISATION_MEAN     = ("CREDIT_UTILISATION",       "mean"),
        CC_RECENT_UTILISATION_MAX      = ("CREDIT_UTILISATION",       "max"),
        CC_RECENT_AMT_DRAWINGS_SUM     = ("AMT_DRAWINGS_CURRENT",     "sum"),
        CC_RECENT_AMT_DRAWINGS_MEAN    = ("AMT_DRAWINGS_CURRENT",     "mean"),
        CC_RECENT_AMT_PAYMENT_SUM      = ("AMT_PAYMENT_TOTAL_CURRENT","sum"),
        CC_RECENT_AMT_PAYMENT_MEAN     = ("AMT_PAYMENT_TOTAL_CURRENT","mean"),
        CC_RECENT_DPD_MAX              = ("SK_DPD",                   "max"),
        CC_RECENT_DPD_MEAN             = ("SK_DPD",                   "mean"),
        CC_RECENT_DPD_FLAG_SUM         = ("DPD_FLAG",                 "sum"),
        CC_RECENT_DPD_FLAG_MEAN        = ("DPD_FLAG",                 "mean"),
        CC_RECENT_DPD_DEF_FLAG_SUM     = ("DPD_DEF_FLAG",             "sum"),
        CC_RECENT_HIGH_UTIL_RATE       = ("HIGH_UTILISATION",         "mean"),
        CC_RECENT_MIN_PAYMENT_RATIO    = ("MIN_PAYMENT_RATIO",        "mean"),
    ).reset_index()

    # ── Merge ─────────────────────────────────────────────────────────────────────
    print("Merging all-history + recent-window features ...")
    agg_final = pd.merge(agg_all, agg_recent, on="SK_ID_CURR", how="left")

    # ── Extra ratio features ──────────────────────────────────────────────────────
    # Total payments vs total drawings — are they paying off what they spend?
    agg_final["CC_PAYMENT_TO_DRAWING_RATIO"] = (
        agg_final["CC_AMT_PAYMENT_TOTAL_SUM"]
        / agg_final["CC_AMT_DRAWINGS_SUM"].replace(0, np.nan)
    )

    # Is recent utilisation higher than all-time? (deterioration signal)
    agg_final["CC_UTILISATION_RECENT_VS_ALL"] = (
        agg_final["CC_RECENT_UTILISATION_MEAN"]
        / agg_final["CC_UTILISATION_MEAN"].replace(0, np.nan)
    )

    # Is recent DPD rate worse than all-time?
    agg_final["CC_DPD_RECENT_VS_ALL_RATIO"] = (
        agg_final["CC_RECENT_DPD_FLAG_MEAN"]
        / agg_final["CC_DPD_FLAG_MEAN"].replace(0, np.nan)
    )

    # ── Summary ───────────────────────────────────────────────────────────────────
    print(f"\n{'─'*55}")
    print(f"  Final shape      : {agg_final.shape}")
    print(f"  Customers        : {agg_final['SK_ID_CURR'].nunique():,}")
    print(f"  Features created : {agg_final.shape[1] - 1}")
    print(f"{'─'*55}")
    print(f"\n  All columns:")
    for col in agg_final.columns:
        print(f"    {col}")

    print(f"\n  Sample (3 rows transposed):")
    print(agg_final.head(3).T.to_string())
    return agg_final

def engineer_application_features(df):
    # Income and credit ratios — strongest single features
    df['CREDIT_INCOME_RATIO'] = df['AMT_CREDIT'] / df['AMT_INCOME_TOTAL']
    df['ANNUITY_INCOME_RATIO'] = df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL']
    df['CREDIT_TERM'] = df['AMT_ANNUITY'] / df['AMT_CREDIT']  # implied loan term
    df['INCOME_PER_PERSON'] = df['AMT_INCOME_TOTAL'] / df['CNT_FAM_MEMBERS']
    df['CREDIT_GOODS_RATIO'] = df['AMT_CREDIT'] / df['AMT_GOODS_PRICE']

    # Age and employment
    df['AGE_YEARS'] = -df['DAYS_BIRTH'] / 365
    df['EMPLOYED_YEARS'] = -df['DAYS_EMPLOYED'] / 365
    df['EMPLOYED_TO_AGE_RATIO'] = df['DAYS_EMPLOYED'] / df['DAYS_BIRTH']

    # External scores — these are pre-computed credit scores and are
    # the most predictive features in the entire dataset
    df['EXT_SOURCE_MEAN'] = df[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].mean(axis=1)
    df['EXT_SOURCE_STD'] = df[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].std(axis=1)
    df['EXT_SOURCE_PROD'] = df['EXT_SOURCE_1'] * df['EXT_SOURCE_2'] * df['EXT_SOURCE_3']

    # Documentation completeness
    doc_cols = [c for c in df.columns if c.startswith('FLAG_DOCUMENT_')]
    df['DOCUMENT_COUNT'] = df[doc_cols].sum(axis=1)

    return df

# To reduce the memory. Not using
def reduce_memory(df):
    for col in df.select_dtypes(include=['float64']).columns:
        df[col] = df[col].astype('float32')
    for col in df.select_dtypes(include=['int64']).columns:
        df[col] = pd.to_numeric(df[col], downcast='integer')
    return df


#load the data
drv = drive.mount('/content/drive')

file_path = '/content/drive/MyDrive/Project#24/data-kaggle/home-credit-default-risk/application_train.csv'
df = pd.read_csv(file_path)
df.head()

bureau_file_path = '/content/drive/MyDrive/Project#24/data-kaggle/home-credit-default-risk/bureau.csv'
bureau_blnc_file_path = '/content/drive/MyDrive/Project#24/data-kaggle/home-credit-default-risk/bureau_balance.csv'
preapp_file_path = '/content/drive/MyDrive/Project#24/data-kaggle/home-credit-default-risk/previous_application.csv'
poscash_file_path = '/content/drive/MyDrive/Project#24/data-kaggle/home-credit-default-risk/POS_CASH_balance.csv'
instpay_file_path = '/content/drive/MyDrive/Project#24/data-kaggle/home-credit-default-risk/installments_payments.csv'
creditbal_file_path = '/content/drive/MyDrive/Project#24/data-kaggle/home-credit-default-risk/credit_card_balance.csv'


dfNan = df.isna().aggregate('sum').sort_values(ascending=True)

print(df.shape)  # ~307,511 rows, 122 columns
print(df['TARGET'].value_counts(normalize=True))

# Critical anomaly fixes
df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace(365243, np.nan)
df = df[df['CODE_GENDER'] != 'XNA']
df = engineer_application_features(df)
bureau__agg_df = aggregate_bureau(bureau_file_path, bureau_blnc_file_path)
preapp_agg_df = aggregate_pre_application(preapp_file_path)
pos_cash_agg_df = pos_cash_aggregate(poscash_file_path)
inst_pay_agg_df = installment_payments_aggregate(instpay_file_path)
creditbal_agg_df = credit_balance_aggregate(creditbal_file_path)

df = df.merge(bureau__agg_df, on='SK_ID_CURR', how='left')
df = df.merge(preapp_agg_df, on='SK_ID_CURR', how='left')
df = df.merge(pos_cash_agg_df, on='SK_ID_CURR', how='left')
df = df.merge(inst_pay_agg_df, on='SK_ID_CURR', how='left')
df = df.merge(creditbal_agg_df, on='SK_ID_CURR', how='left')

print(bureau__agg_df.head())
print(preapp_agg_df.head())
print(pos_cash_agg_df.head())

print(df.describe())
print(df.shape)
print(df.info())

df.to_csv('/content/drive/MyDrive/Project#24/data-kaggle/dd  f  /feature_engineered_data.csv')

#dfNan.head(10)
#dfNan.sort_values(ascending=False)
#plt.bar(dfNan.values, dfNan.index)
#plt.xticks(rotation=90)
#plt.show()

#fig, ax = plt.subplots()
#fig.set_size_inches(w=20, h=200)
#width = 2 # the width of the bars
#ind = np.arange(len(dfNan))  # the x locations for the groups
#ax.barh(ind, dfNan.values, width, color='red')
#ax.set_yticks(ind+width/2)

#print(f'index : {ind}, yticks : {ind+width/2}')
#ax.set_yticklabels(dfNan.index, minor=True)
#plt.title('Nan Values in Columns')
#plt.xlabel('Values')
#plt.ylabel('Columns')
#bars = ax.bar(dfNan.values, dfNan.index)

# Add labels to bars
#ax.bar_label(bars, label_type='edge', color='blue')
#plt.show()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
(307511, 122)
TARGET
0    0.919271
1    0.080729
Name: proportion, dtype: float64
Approved rows  : 1,036,781
Cancelled rows : 316,319

Aggregating approved applications ...
Aggregating cancelled applications ...

Merging approved & cancelled features ...

Final aggregated shape : (338152, 39)
Customers covered      : 338,152

Columns:
['SK_ID_CURR', 'APPROVED_COUNT', 'APPROVED_TOTAL_CREDIT', 'APPROVED_AVG_CREDIT', 'APPROVED_MAX_CREDIT', 'APPROVED_MIN_CREDIT', 'APPROVED_TOTAL_APPLICATION', 'APPROVED_AVG_APPLICATION', 'APPROVED_TOTAL_ANNUITY', 'APPROVED_AVG_ANNUITY', 'APPROVED_AVG_DOWN_PAYMENT', 'APPROVED_AVG_GOODS_PRICE', 'APPROVED_AVG_CNT_PAYMENT', 'APPROVED_MAX_CNT_PAYMENT', 'APPROVED_AVG_DAYS_DECISION', 'APPROVED_MIN_DAYS_DECISION', 'APPROVED_SUM_INSURED', 'APPROVED_AVG_RATE_DOWN_PAYMENT', 'APPROVED_AVG_RATE_INTEREST_PRIM', 'APPROVED_CREDIT_TO_APPLICATION_R